In [1]:
pip install geopy


Defaulting to user installation because normal site-packages is not writeable
  Using cached geopy-2.4.1-py3-none-any.whl.metadata (6.8 kB)
Using cached geopy-2.4.1-py3-none-any.whl (125 kB)

   ---------------------------------------- 0/2 [geographiclib]
   ---------------------------------------- 0/2 [geographiclib]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]

Note: y


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import time
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

VENUE_CSV = "data_processed/venue_master.csv"
MATCH_CSV = "data_processed/venue_match_level.csv"
CACHE_CSV = "data_processed/venue_geo_cache.csv"

BATCH_SIZE = 20   # <-- key change

venue_df = pd.read_csv(VENUE_CSV)
match_df = pd.read_csv(MATCH_CSV)


# LOAD CACHE

if os.path.exists(CACHE_CSV):
    cache_df = pd.read_csv(CACHE_CSV)
    geo_cache = dict(
        zip(cache_df["venue"], zip(cache_df["latitude"], cache_df["longitude"]))
    )
else:
    geo_cache = {}


# GEOCODER

geolocator = Nominatim(user_agent="cricket_venue_geocoder", timeout=5)

def safe_geocode(query):
    try:
        return geolocator.geocode(query)
    except (GeocoderTimedOut, GeocoderUnavailable):
        return None


# FIND UNPROCESSED VENUES
remaining = venue_df[~venue_df["venue"].isin(geo_cache)]

print(f"Remaining venues to geocode: {len(remaining)}")

if remaining.empty:
    print(" All venues already geocoded")
    exit()


# PROCESS ONE BATCH ONLY

batch = remaining.head(BATCH_SIZE)

for _, row in batch.iterrows():
    venue = row["venue"]
    city = row["city"]

    queries = [
        f"{venue}, {city}, cricket stadium",
        f"{venue}, {city}",
        venue,
        city
    ]

    location = None
    for q in queries:
        location = safe_geocode(q)
        if location:
            break

    if location:
        lat, lon = location.latitude, location.longitude
        print(f" {venue}")
    else:
        lat, lon = None, None
        print(f" Skipped: {venue}")

    geo_cache[venue] = (lat, lon)
    time.sleep(2)   


# SAVE CACHE

pd.DataFrame(
    [{"venue": v, "latitude": lat, "longitude": lon}
     for v, (lat, lon) in geo_cache.items()]
).to_csv(CACHE_CSV, index=False)

print("Batch complete. Re-run script to continue.")


# APPLY TO CSVs

venue_df = venue_df.merge(
    pd.DataFrame(
        [{"venue": v, "latitude": lat, "longitude": lon}
         for v, (lat, lon) in geo_cache.items()]
    ),
    on="venue",
    how="left"
)

venue_df.to_csv(VENUE_CSV, index=False)

match_df = match_df.merge(
    venue_df[["venue", "latitude", "longitude"]],
    on="venue",
    how="left"
)

match_df.to_csv(MATCH_CSV, index=False)


Remaining venues to geocode: 11
✔ Colombo Cricket Club Ground
✔ Stars Arena Hofstade, Zemst
✔ Titwood, Glasgow
✔ Ballpark Ground, Graz
✔ Grainville, St Saviour
✔ Ishoj Cricket Club, Vejledalen
✔ Koge Cricket Club
✔ Maple Leaf North-East Ground, King City
✔ Sportpark Westvliet, Voorburg
✔ Vassil Levski National Sports Academy, Sofia
✔ Estonian National Cricket and Rugby Field
Batch complete. Re-run script to continue.


MergeError: Passing 'suffixes' which cause duplicate columns {'latitude_x', 'longitude_x'} is not allowed.

In [ ]:
# sync code

import pandas as pd

VENUE_CSV = "data_processed/venue_master.csv"
MATCH_CSV = "data_processed/venue_match_level.csv"
CACHE_CSV = "data_processed/venue_geo_cache.csv"




venue_df = pd.read_csv(VENUE_CSV)
match_df = pd.read_csv(MATCH_CSV)
cache_df = pd.read_csv(CACHE_CSV)


venue_df = venue_df.drop(
    columns=[
        "latitude", "longitude",
        "latitude_x", "longitude_x",
        "latitude_y", "longitude_y"
    ],
    errors="ignore"
)


venue_df = venue_df.merge(
    cache_df,
    on="venue",
    how="left"
)

venue_df.to_csv(VENUE_CSV, index=False)
print(" venue_master.csv fully synced from cache")


match_df = match_df.drop(
    columns=[
        "latitude", "longitude",
        "latitude_x", "longitude_x",
        "latitude_y", "longitude_y"
    ],
    errors="ignore"
)

# Attach lat/lon via venue
match_df = match_df.merge(
    venue_df[["venue", "latitude", "longitude"]],
    on="venue",
    how="left"
)

match_df.to_csv(MATCH_CSV, index=False)
print(" venue_match_level.csv fully synced from venue_master")


✅ venue_master.csv fully synced from cache
✅ venue_match_level.csv fully synced from venue_master


Removing repetitive names

In [ ]:
import pandas as pd
import re
from math import radians, sin, cos, sqrt, atan2
from collections import defaultdict

INPUT_CSV = "data_processed/venue_master.csv"
OUT_LOOKUP = "data_processed/venue_lookup.csv"
OUT_REVIEW = "data_processed/venue_lookup_review.csv"


# Utilities

def normalize_name(name, city=None):
    if pd.isna(name):
        return ""

    name = str(name).lower()

    if city is not None and not pd.isna(city):
        name = name.replace(str(city).lower(), "")

    name = re.sub(r"[^\w\s]", " ", name)

    noise = [
        "stadium", "ground", "cricket", "international",
        "sports", "complex", "oval"
    ]
    for n in noise:
        name = name.replace(n, "")

    name = re.sub(r"\s+", " ", name).strip()
    return name

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dl = radians(lon2 - lon1)

    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dl/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

def make_venue_id(name):
    return re.sub(r"\W+", "_", str(name).upper()).strip("_")

def coord_key(lat, lon, precision=4):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return (round(lat, precision), round(lon, precision))


# Load data

df = pd.read_csv(INPUT_CSV)

df["norm_name"] = df.apply(
    lambda r: normalize_name(r["venue"], r["city"]),
    axis=1
)


# name + city

groups = df.groupby(["norm_name", "city"], dropna=False)

lookup_rows = []
review_rows = []

for (norm, city), g in groups:
    if len(g) == 1:
        r = g.iloc[0]
        lookup_rows.append({
            "venue_id": make_venue_id(r["venue"]),
            "canonical_name": r["venue"],
            "city": r["city"],
            "latitude": r["latitude"],
            "longitude": r["longitude"],
            "aliases": r["venue"]
        })
        continue

    base = g.iloc[0]
    same_bucket = [base]
    conflicts = []

    for _, r in g.iloc[1:].iterrows():
        if pd.notna(base["latitude"]) and pd.notna(r["latitude"]):
            d = haversine(
                base["latitude"], base["longitude"],
                r["latitude"], r["longitude"]
            )
            if d <= 150:
                same_bucket.append(r)
            else:
                conflicts.append(r)
        else:
            conflicts.append(r)

    canonical = same_bucket[0]["venue"]
    venue_id = make_venue_id(canonical)

    lookup_rows.append({
        "venue_id": venue_id,
        "canonical_name": canonical,
        "city": city,
        "latitude": same_bucket[0]["latitude"],
        "longitude": same_bucket[0]["longitude"],
        "aliases": "; ".join(sorted(set(r["venue"] for r in same_bucket)))
    })

    for r in conflicts:
        review_rows.append({
            "candidate_venue": r["venue"],
            "city": city,
            "possible_match": canonical,
            "reason": "Name mismatch or coordinate distance >150m / missing"
        })


# coordinate collapse

coord_groups = defaultdict(list)

for row in lookup_rows:
    key = (row["city"], coord_key(row["latitude"], row["longitude"]))
    coord_groups[key].append(row)

final_lookup = []

for (_, _), rows in coord_groups.items():
    if len(rows) == 1:
        final_lookup.append(rows[0])
        continue

    canonical = rows[0]

    all_aliases = []
    for r in rows:
        all_aliases.extend(r["aliases"].split("; "))

    final_lookup.append({
        "venue_id": canonical["venue_id"],
        "canonical_name": canonical["canonical_name"],
        "city": canonical["city"],
        "latitude": canonical["latitude"],
        "longitude": canonical["longitude"],
        "aliases": "; ".join(sorted(set(all_aliases)))
    })


# Save outputs

pd.DataFrame(final_lookup).to_csv(OUT_LOOKUP, index=False)
pd.DataFrame(review_rows).to_csv(OUT_REVIEW, index=False)

print(" venue_lookup.csv generated (final, merged)")
print(" venue_lookup_review.csv generated (manual review only)")


✅ venue_lookup.csv generated (final, merged)
⚠️ venue_lookup_review.csv generated (manual review only)


Hard mapping the statiums

In [ ]:
import pandas as pd
import re
from collections import defaultdict

INPUT_CSV = "data_processed/venue_master.csv"
OUT_LOOKUP = "data_processed/venue_lookup.csv"

COORD_PRECISION = 4  # ~11 meters

def make_venue_id(name):
    return re.sub(r"\W+", "_", str(name).upper()).strip("_")

def coord_key(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    return (round(lat, COORD_PRECISION), round(lon, COORD_PRECISION))


df = pd.read_csv(INPUT_CSV)


df = df.dropna(subset=["latitude", "longitude"])


coord_groups = defaultdict(list)

for _, r in df.iterrows():
    key = coord_key(r["latitude"], r["longitude"])
    coord_groups[key].append(r)

lookup_rows = []

for (latlon), rows in coord_groups.items():

    canonical_row = max(rows, key=lambda r: len(str(r["venue"])))

    aliases = sorted(set(str(r["venue"]) for r in rows))

    lookup_rows.append({
        "venue_id": make_venue_id(canonical_row["venue"]),
        "canonical_name": canonical_row["venue"],
        "city": canonical_row["city"],
        "latitude": canonical_row["latitude"],
        "longitude": canonical_row["longitude"],
        "aliases": "; ".join(aliases)
    })


lookup_df = pd.DataFrame(lookup_rows)
lookup_df.to_csv(OUT_LOOKUP, index=False)

print(f" venue_lookup.csv generated using HARD coordinate merge")
print(f"Venues before: {len(df)}")
print(f"Venues after : {len(lookup_df)}")


✅ venue_lookup.csv generated using HARD coordinate merge
Venues before: 445
Venues after : 281
